# 03 · Edge AI Triage — flag suspicious tiles *while* scanning

**Problem.** Keeping full z-stacks for every tile multiplies storage, but throwing them away loses detail a pathologist
may need. Scanners with on-board GPUs can run AI **during acquisition**, analysing each field of view as it arrives.

**Approach.** A MobileNetV3-Small classifier (PatchCamelyon: tumour vs. normal) runs on every tile as it is scanned.
Tiles it flags keep their z-stack; the rest keep only the fused image. The model is exported to **ONNX**, quantised to
**INT8**, and benchmarked on CPU — no PyTorch at inference time, like an embedded scanner PC.

In [ ]:
# --- setup: works locally (run from repo root or notebooks/) and on Google Colab ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("wsi-smart-scan"):
    !git clone https://github.com/Brajesh250/wsi-smart-scan.git
    %cd wsi-smart-scan
    !pip -q install -r requirements.txt
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
os.makedirs("results", exist_ok=True); os.makedirs("models", exist_ok=True); os.makedirs("assets", exist_ok=True)

In [ ]:
import torch, h5py
from torch.utils.data import DataLoader, Subset
from src.triage import (PCamDataset, build_model, train_triage, evaluate_triage, predict_probs, export_onnx,
                        quantize_int8, benchmark, OnnxTriage)
torch.manual_seed(0)
QUICK = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
P = "data/pcam/camelyonpatch_level_2_split_{}_{}.h5"
print("device:", DEVICE)

## 1 · Data

If the 6 GB train split was downloaded (`--pcam-train`) we train on it; otherwise we train on 80 % of the 32k validation
split. The official **test split** (32,768 patches from different slides) is used only for final evaluation.

In [ ]:
if os.path.exists(P.format("train", "x")):
    train_ds = PCamDataset(P.format("train", "x"), P.format("train", "y"), train=True, limit=2000 if QUICK else 100000)
    val_ds = PCamDataset(P.format("valid", "x"), P.format("valid", "y"), limit=500 if QUICK else 8000)
else:
    full = PCamDataset(P.format("valid", "x"), P.format("valid", "y"), train=True, limit=2000 if QUICK else None)
    idx = np.random.default_rng(0).permutation(len(full)); cut = int(0.8 * len(full))
    train_ds = Subset(full, idx[:cut])
    val_ds = Subset(PCamDataset(P.format("valid", "x"), P.format("valid", "y"), limit=len(full)), idx[cut:])
test_ds = PCamDataset(P.format("test", "x"), P.format("test", "y"), limit=1000 if QUICK else None)
print(len(train_ds), len(val_ds), len(test_ds))
kw = dict(num_workers=2, pin_memory=DEVICE == "cuda")
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, **kw)
val_dl = DataLoader(val_ds, batch_size=256, **kw)
test_dl = DataLoader(test_ds, batch_size=256, **kw)

## 2 · Train MobileNetV3-Small (ImageNet-pretrained)

In [ ]:
model = build_model(pretrained=not QUICK)
print("parameters:", sum(p.numel() for p in model.parameters()))
history = train_triage(model, train_dl, val_dl, epochs=1 if QUICK else 4, lr=1e-3, device=DEVICE)
torch.save(model.state_dict(), "models/triage.pt")

## 3 · Test-set evaluation

In [ ]:
from sklearn.metrics import roc_curve
test_metrics = evaluate_triage(model, test_dl, DEVICE)
print(test_metrics)
p, y = predict_probs(model, test_dl, DEVICE)
fpr, tpr, thr = roc_curve(y, p)
# operating point for triage: we prefer high recall (don't drop z-stacks of tumour tiles)
op = np.argmax(tpr >= 0.95); triage_threshold = float(thr[op])
print(f"threshold for 95% recall: {triage_threshold:.3f}  (FPR {fpr[op]:.3f})")
plt.figure(figsize=(4.5, 4)); plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "k--")
plt.scatter(fpr[op], tpr[op], c="r", zorder=3, label="95% recall operating point")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"PCam test ROC (AUC {test_metrics['auc']:.3f})"); plt.legend()
plt.tight_layout(); plt.savefig("assets/triage_roc.png", dpi=120); plt.show()

## 4 · Export → ONNX → INT8 and benchmark on CPU

In [ ]:
export_onnx(model, "models/triage_fp32.onnx")
with h5py.File(P.format("valid", "x")) as f:
    calib = f["x"][:256]                                         # calibration tiles for static quantisation
quantize_int8("models/triage_fp32.onnx", "models/triage_int8.onnx", calib_tiles=calib)

with h5py.File(P.format("test", "x")) as f, h5py.File(P.format("test", "y")) as g:
    n = 500 if QUICK else 4000
    xs, ys = f["x"][:n], g["y"][:n].reshape(-1)
from sklearn.metrics import roc_auc_score
rows = []
for path in ["models/triage_fp32.onnx", "models/triage_int8.onnx"]:
    probs = np.concatenate([OnnxTriage(path)(xs[i:i+256]) for i in range(0, n, 256)])
    rows.append({**benchmark(path, runs=50 if QUICK else 300), "auc": roc_auc_score(ys, probs),
                 "acc": float(((probs > 0.5) == ys).mean())})
edge = pd.DataFrame(rows).round(4)
edge.to_csv("results/edge_benchmark.csv", index=False)
edge

## 5 · Streaming triage inside the smart scan

We build a mosaic "slide" from **labelled PCam test patches** (so the ground truth is known), stream it through the
scanner, and measure: how many tumour tiles kept their z-stack (recall) and how much storage was saved.

In [ ]:
from src.pipeline import SmartScanner, ScanConfig
G, T = 8, 96
with h5py.File(P.format("test", "x")) as f, h5py.File(P.format("test", "y")) as g:
    start = 0 if QUICK else 5000
    tiles, labels = f["x"][start:start + G*G], g["y"][start:start + G*G].reshape(G, G)
mosaic = tiles.reshape(G, G, T, T, 3).transpose(0, 2, 1, 3, 4).reshape(G*T, G*T, 3)

cfg = ScanConfig(tile=T, triage_model="models/triage_int8.onnx", triage_threshold=triage_threshold,
                 failure_region=False, min_tissue=0.0)
scanner = SmartScanner(cfg)
rep = scanner.run(mosaic)
keep = scanner.last["keep"]
recall = (keep & (labels == 1)).sum() / max((labels == 1).sum(), 1)
print(f"tumour tiles: {(labels==1).sum()} | z-stack kept for {recall:.0%} of them")
print(f"z-stacks kept: {rep['tiles_zstack_kept']}/{rep['tiles_tissue']} | storage saved: {rep['storage_saved_pct']}%")
scanner.save_figures("results/triage_demo")

fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
ax[0].imshow(mosaic); ax[0].set_title("Mosaic of PCam test tiles (red = true tumour)")
for r, c in np.argwhere(labels == 1):
    ax[0].add_patch(plt.Rectangle((c*T, r*T), T, T, fill=False, ec="red", lw=1.5))
from src.utils import overlay_heatmap, to_uint8
ax[1].imshow(overlay_heatmap(mosaic, scanner.last["triage"], T, cmap="magma", vmin=0, vmax=1))
ax[1].set_title("Edge triage: tumour probability while scanning")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig("assets/triage_stream.png", dpi=120); plt.show()

json.dump({"test": test_metrics, "triage_threshold": triage_threshold, "edge": rows,
           "stream": {"tumour_recall": float(recall), "storage_saved_pct": rep["storage_saved_pct"]}},
          open("results/triage_results.json", "w"), indent=2, default=float)